# Sharp India — Smart Sales & Support Agent

## Problem Statement

**Sharp Business Systems (India) Ltd.** sells MFPs, Interactive Displays, Air Purifiers & Office Automation solutions.

Build an intelligent agent that uses **3 types of tools**:

| # | Tool Type | Purpose |
|---|---|---|
| 1 | **DuckDuckGo Search** | Search web for Sharp India news, service centers, product availability |
| 2 | **Retriever Tool** | Answer from internal product KB (specs, warranty, return policy) |
| 3 | **Agent as Tool** | Complaint resolution sub-agent with classify → resolve → ticket flow |

In [ ]:
!pip install langchain langchain-openai langchain-community langgraph duckduckgo-search faiss-cpu

In [ ]:
from google.colab import userdata
import os
os.environ["AZURE_OPENAI_API_KEY"] = userdata.get('AZURE_OPENAI_API_KEY')
os.environ["AZURE_OPENAI_ENDPOINT"] = userdata.get('AZURE_OPENAI_ENDPOINT')
os.environ["OPENAI_API_VERSION"] = "2025-03-01-preview"

In [ ]:
from langchain_openai import AzureChatOpenAI

model = AzureChatOpenAI(
    model="gpt-4.1-mini",
    azure_deployment="gpt-4.1-mini"
)

---
## Tool 1: DuckDuckGo Web Search
Use this for real-time info — service centers, latest products, news about Sharp India.

In [ ]:
from langchain_community.tools import DuckDuckGoSearchRun

search_tool = DuckDuckGoSearchRun()

---
## Tool 2: Retriever Tool (Product Knowledge Base)
Load Sharp India product docs into a vectorstore and create a retriever tool.

In [ ]:
from langchain_community.vectorstores import FAISS
from langchain_openai import AzureOpenAIEmbeddings
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.tools.retriever import create_retriever_tool

# Sharp India product knowledge base
sharp_kb = """
Sharp BP-70M45: A3 monochrome MFP, 45 ppm, 100-sheet DSPF, 1200x1200 dpi, PCL6/PS3,
standard duplex, network print/scan/copy. Warranty: 1 year or 1 lakh copies whichever is earlier.
Price: Contact dealer. Ideal for large offices with heavy print volumes.

Sharp BP-50M26: A3 monochrome MFP, 26 ppm, 100-sheet DSPF, standard duplex,
USB/Network connectivity. Warranty: 1 year or 60,000 copies whichever is earlier.
Suitable for small to medium offices.

Sharp BP-70C65: A3 color MFP, 65 ppm color/mono, 10.1 inch touchscreen,
standard wireless LAN, 100-sheet DSPF. Warranty: 1 year or 1.5 lakh copies.
Best for marketing teams and design studios.

Sharp BIG PAD PN-L752B: 75-inch 4K interactive display, 10-point multi-touch,
built-in whiteboard software, wireless screen sharing (up to 50 devices),
HDMI x3, USB-C. Ideal for meeting rooms and classrooms. Warranty: 3 years onsite.

Sharp BIG PAD PN-L652B: 65-inch 4K interactive display, 10-point touch,
built-in whiteboard, wireless sharing. Warranty: 3 years onsite.
Compact option for huddle rooms.

Sharp FP-J80M: Air purifier with Plasmacluster technology, covers up to 620 sq ft,
True HEPA + Carbon filter, 3 fan speeds, auto mode, humidity sensor.
Warranty: 2 years on unit, 1 year on filter. Price: approx INR 38,000.

Sharp FP-J60M: Air purifier with Plasmacluster, covers up to 450 sq ft,
HEPA filter, 3 speeds. Warranty: 2 years. Price: approx INR 28,000.

Annual Maintenance Contract (AMC): Available for all MFPs after warranty period.
Includes quarterly servicing, toner replacement, and parts. Contact regional office for pricing.

Return Policy: Products can be returned within 7 days of delivery if unopened and in original packaging.
Opened products are eligible for replacement only if manufacturing defect is found within 48 hours.
Refunds processed within 10 business days to original payment method.

Service Centers:
- Mumbai: Andheri West, near Lokhandwala. Ph: 022-XXXXXXXX
- Delhi: Nehru Place, Ground Floor. Ph: 011-XXXXXXXX
- Bangalore: Koramangala, 4th Block. Ph: 080-XXXXXXXX
- Chennai: T Nagar, Pondy Bazaar Road. Ph: 044-XXXXXXXX
- Hyderabad: Ameerpet, beside Metro Station. Ph: 040-XXXXXXXX
- Kolkata: Salt Lake, Sector V. Ph: 033-XXXXXXXX
"""

# Split and embed
splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)
docs = splitter.create_documents([sharp_kb])

embeddings = AzureOpenAIEmbeddings(
    model="text-embedding-3-small",
    azure_deployment="text-embedding-3-small"
)

vectorstore = FAISS.from_documents(docs, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

retriever_tool = create_retriever_tool(
    retriever,
    name="sharp_product_kb",
    description="Search Sharp India internal knowledge base for product specs, warranty info, return policy, AMC details, and service center locations."
)

---
## Tool 3: Agent as Tool — Complaint Resolution Sub-Agent
A specialized agent that classifies complaints, provides resolution steps, and generates tickets.

In [ ]:
from langchain.tools import tool
from langchain.agents import create_agent
import random
from datetime import datetime


@tool
def classify_complaint(description: str) -> str:
    """Classify a customer complaint into a category: hardware, software, delivery, or billing."""
    description_lower = description.lower()
    if any(w in description_lower for w in ["jam", "noise", "broken", "not working", "damage", "hardware", "physical"]):
        return "hardware"
    elif any(w in description_lower for w in ["error", "software", "update", "screen freeze", "driver", "connectivity"]):
        return "software"
    elif any(w in description_lower for w in ["delivery", "shipping", "wrong model", "not received", "late", "package"]):
        return "delivery"
    elif any(w in description_lower for w in ["bill", "charge", "invoice", "payment", "refund", "overcharged"]):
        return "billing"
    else:
        return "hardware"


@tool
def get_resolution(category: str, product: str) -> str:
    """Get resolution steps for a complaint based on category and product name."""
    resolutions = {
        "hardware": f"""Resolution for {product} (Hardware Issue):
1. Power off the device completely
2. Check for any visible obstructions or damage
3. Clean the device as per user manual instructions
4. Power on and test again
5. If issue persists, contact nearest Sharp service center for onsite repair""",
        "software": f"""Resolution for {product} (Software Issue):
1. Restart the device
2. Check for firmware updates on sharp.co.in/support
3. Reset to factory settings if needed (backup data first)
4. Reinstall drivers from official Sharp India website
5. If unresolved, request remote diagnostic support""",
        "delivery": f"""Resolution for {product} (Delivery Issue):
1. Verify order details and tracking number
2. If wrong model received — DO NOT open packaging
3. Raise return request within 7 days
4. Pickup will be scheduled within 2 business days
5. Correct product will be dispatched after pickup confirmation""",
        "billing": f"""Resolution for {product} (Billing Issue):
1. Cross-check invoice with order confirmation email
2. If overcharged, raise dispute with order ID
3. Refund (if applicable) processed in 10 business days
4. For EMI queries, contact finance partner directly"""
    }
    return resolutions.get(category, "Please contact Sharp India support at 1800-XXX-XXXX.")


@tool
def generate_ticket(customer_name: str, product: str, issue: str) -> str:
    """Generate a complaint ticket with unique ID for tracking."""
    ticket_id = f"SHARP-TKT-{datetime.now().strftime('%Y%m%d')}-{random.randint(1000, 9999)}"
    return f"""Ticket Created Successfully!
- Ticket ID: {ticket_id}
- Customer: {customer_name}
- Product: {product}
- Issue: {issue}
- Status: OPEN
- Expected Response: Within 24 hours
Please save your Ticket ID for future reference."""

In [ ]:
# Create the complaint resolution sub-agent
complaint_agent = create_agent(
    model,
    tools=[classify_complaint, get_resolution, generate_ticket],
    system_prompt="""You are Sharp India's complaint resolution specialist.
When a customer describes a problem:
1. First use classify_complaint to determine the category
2. Then use get_resolution with the category and product to get steps
3. Finally use generate_ticket to create a tracking ticket
Return all information clearly to the customer."""
)

# Wrap sub-agent as a tool for the main agent
@tool
def complaint_resolution_agent(complaint: str) -> str:
    """Delegate customer complaints to the complaint resolution specialist agent.
    Use this when a customer reports a problem, defect, or issue with their Sharp product."""
    response = complaint_agent.invoke({"messages": [{"role": "user", "content": complaint}]})
    return response["messages"][-1].content

---
## Main Agent: Sharp India Sales & Support Assistant
Combines all 3 tools with short-term memory.

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver

sharp_agent = create_agent(
    model,
    tools=[search_tool, retriever_tool, complaint_resolution_agent],
    system_prompt="""You are Sharp India's intelligent sales and support assistant.

You have 3 tools:
1. duckduckgo_search - For real-time web search (latest news, availability, competitors)
2. sharp_product_kb - For internal product specs, warranty, return policy, service centers
3. complaint_resolution_agent - For handling customer complaints (classify, resolve, ticket)

Rules:
- For product specs/warranty/policy questions → use sharp_product_kb FIRST
- For latest news/availability/general web info → use duckduckgo_search
- For complaints/issues/defects → use complaint_resolution_agent
- Always be polite and professional
- Remember what the customer told you earlier in the conversation""",
    checkpointer=InMemorySaver(),
)

---
## Test the Agent

In [ ]:
thread = {"configurable": {"thread_id": "customer-001"}}

# Test 1: Retriever Tool — product specs
response = sharp_agent.invoke(
    {"messages": [{"role": "user", "content": "What is the warranty on Sharp BIG PAD PN-L752B?"}]},
    thread,
)
print("🤖", response["messages"][-1].content)

In [ ]:
# Test 2: Web Search Tool — latest info
response = sharp_agent.invoke(
    {"messages": [{"role": "user", "content": "What are the latest Sharp multifunction printers available in India?"}]},
    thread,
)
print("🤖", response["messages"][-1].content)

In [ ]:
# Test 3: Complaint Agent — hardware issue
response = sharp_agent.invoke(
    {"messages": [{"role": "user", "content": "My name is Raj. My Sharp BP-70M45 keeps showing paper jam error even after clearing. Please help."}]},
    thread,
)
print("🤖", response["messages"][-1].content)

In [ ]:
# Test 4: Memory — agent should remember the customer name and product
response = sharp_agent.invoke(
    {"messages": [{"role": "user", "content": "Where is the nearest service center for me in Mumbai?"}]},
    thread,
)
print("🤖", response["messages"][-1].content)

---
## Streaming: Print Agent Steps While Running

In [ ]:
from langchain_core.messages import AIMessage, HumanMessage, ToolMessage

thread2 = {"configurable": {"thread_id": "customer-002"}}

for chunk in sharp_agent.stream(
    {"messages": [{"role": "user", "content": "I ordered a BIG PAD last week but received the wrong model. What should I do?"}]},
    thread2,
    stream_mode="values",
):
    msg = chunk["messages"][-1]
    if isinstance(msg, AIMessage):
        if msg.tool_calls:
            print(f"\U0001f527 Calling: {[tc['name'] for tc in msg.tool_calls]}")
        elif msg.content:
            print(f"\U0001f916 Agent: {msg.content}")
    elif isinstance(msg, ToolMessage):
        print(f"\U0001f4ce Tool response: {msg.content[:200]}...")